# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a guided workflow for loading and exploring the FAIR^2 dataset on protein abundance and modifications in human mast cells, using the `mlcroissant` library.

#### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` is installed
!pip install -q mlcroissant

## 1. Data Loading

We load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Show dataset overview
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview

Let’s review available record sets, their fields, and identify their `@id`s.

**Note:** All identifiers are referenced by their `@id` per the Croissant spec.

In [ ]:
# List all record sets and their fields by @id

record_sets = [r for r in metadata.record_sets]
print("Available Record Sets and their Fields:\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs.id}")
    print(f"  Name: {rs.name}")
    if hasattr(rs, 'description'):
        print(f"  Description: {rs.description}")
    print("  Fields (with @id):")
    for f in rs.fields:
        print(f"    - {f.id} (name: {f.name}, type: {f.data_type})")
    print("---")

### Inspecting a single record
Let’s take a look at the first record (row) in one record set to get a sense of the data structure. We will reference record sets by their `@id`.

In [ ]:
# For example, print the first record of the first record set

first_rs_id = record_sets[0].id

for i, record in enumerate(dataset.records(record_set=first_rs_id)):
    print(f"Record {i}: {json.dumps(record, indent=2)}")
    if i >= 0:
        break

## 3. Data Extraction

Let’s extract data from each available record set into a pandas DataFrame for analysis.

In [ ]:
# Extract data from all record sets using their @id

dataframes = {}
for rs in record_sets:
    print(f"Loading records from RecordSet @id: {rs.id} (name: {rs.name})...")
    records = list(dataset.records(record_set=rs.id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs.id] = df
        print(f"  Columns: {df.columns.tolist()}")
        print(f"  Preview:\n{df.head(2)}\n---\n")
    else:
        print("  No records found in this set.\n---\n")

## 4. Exploratory Data Analysis (EDA)

We'll demonstrate common data analyses: filtering, normalization, and grouping. All fields are referenced by their `@id` as per Croissant conventions.

Let’s pick a numeric field and a group/categorization field from the first non-empty record set (by examining available columns). Adjust the fields below to match actual column `@id`s in your data for your own purposes.

In [ ]:
# Example: EDA on the first record set with data

# Find the first dataframe that has data
for rs_id, df in dataframes.items():
    if not df.empty:
        base_record_set_id = rs_id
        print(f"Using RecordSet: {base_record_set_id}")
        print("Columns:", df.columns.tolist())
        display(df.head(2))
        break

# You may need to change these field ids below based on the record set structure
# Let's try to select a numeric and a group field automatically
df = dataframes[base_record_set_id]

# Try to infer a numeric field
numeric_field = None
possible_numeric_types = [int, float]

for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field = col
        break

if not numeric_field:
    # Attempt to convert columns to numeric and find the first that works
    for col in df.columns:
        try:
            converted = pd.to_numeric(df[col], errors='coerce')
            if converted.notnull().sum() > 0:
                df[col] = converted
                numeric_field = col
                break
        except Exception:
            continue

print(f"Selected numeric field (by @id): {numeric_field}")

# Try to find a group field (categorical/string)
group_field = None
for col in df.columns:
    if col != numeric_field and df[col].dtype == object:
        group_field = col
        break
print(f"Selected group field (by @id): {group_field}")

if numeric_field:
    # Drop NaN from numeric_field for EDA
    non_null_df = df.dropna(subset=[numeric_field])

    # Set a threshold
    threshold = non_null_df[numeric_field].mean()  # Mean as example

    filtered_df = non_null_df[non_null_df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    print(filtered_df.head())

    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, norm_col]].head())

    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().sort_values(numeric_field, ascending=False)
        print(f"Grouped mean of {numeric_field} by {group_field}:")
        print(grouped_df.head())
else:
    print("No numeric field found for EDA demo.")

## 5. Visualization

Let's visualize the distribution of the selected numeric field and its relationship to the group field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

if numeric_field:
    # Histogram for numeric field
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # Boxplot by group field
    if group_field and group_field in df.columns:
        plt.figure(figsize=(10,5))
        # Limit to top 10 groups for clarity
        top_groups = df[group_field].value_counts().index[:10]
        sns.boxplot(x=group_field, y=numeric_field, data=df[df[group_field].isin(top_groups)])
        plt.xticks(rotation=45, ha='right')
        plt.title(f"{numeric_field} by {group_field} (Top 10 groups)")
        plt.show()

## 6. Conclusion

In this notebook, we have loaded and programmatically explored the FAIR^2 dataset for protein abundance and modification analysis. Using Croissant `@id` references, we've:
- Loaded dataset metadata and inspected schema structure
- Extracted records for each record set into DataFrames
- Performed filtering, normalization, and grouped analysis on a selected numeric field
- Visualized distributions and groupwise summaries

You can extend this analysis to specific biological hypotheses or data quality audits. Change field `@id`s in the analysis steps to focus on particular variables of interest from the schema. For more, see the [mlcroissant documentation](https://mlcommons.github.io/croissant/).
